# 09 — Fastlog / Sparse Predicate Recording

**Purpose:** Show the sparse predicate recording workflow: `tl.record(save=...)` captures
only the ops that match a predicate, with far lower overhead than a full trace. Show what
a `Recording` looks like, how to inspect its event stream, how to cook it into a full
`Trace` via `to_trace()`, and edge-case behaviors: `halt`, module event visibility,
and `tl.fastlog.dry_run` for pre-flight label discovery.

**Surfaces covered:**
- [ ] `tl.record(model, x, save=...)` — sparse predicate recording entry point
- [ ] `Recording.__repr__` / `Recording.summary()` — overview of what was captured
- [ ] `Recording.n_ops` / `Recording.n_records` — counts
- [ ] `Recording.activation_payloads_by_raw_label` — tensor payloads
- [ ] `Recording.records` — `ActivationRecord` list; `.ctx.label`, `.ctx.func_name`
- [ ] `Recording.to_pandas()` — tabular view of recorded events
- [ ] `Recording.to_trace()` — cook sparse recording into a full `Trace`
- [ ] `Recording.recording_trace` — the underlying `RecordingTrace` event stream
- [ ] `RecordingTrace.to_pandas()` — full event table (ops + module enter/exit)
- [ ] `RecordingTrace.summary()` — counts by event kind
- [ ] `tl.fastlog.dry_run(model, x, save=...)` — pre-flight label discovery
- [ ] `RecordingTrace.repredicate(other_keep_op=...)` — re-apply a predicate offline
- [ ] `halt=` kwarg — early-exit predicate on the recording
- [ ] `Recording.halted` / `Recording.halt_reason` — inspect halt result
- [ ] `tl.fastlog` module attributes (shim, types, helpers)
- [ ] removed `keep_op=` / `keep_module=` aliases raise TypeError (`save=` is the only form)

## 1. Setup

In [ ]:
import pathlib
import sys

# Pin imports to THIS checkout: the notebook dir (for _models.py) plus the repo
# root, so `import torchlens` audits the code this notebook ships with -- not a
# pip-installed copy from another checkout.
_NB_DIR = pathlib.Path.cwd()
if not (_NB_DIR / "_models.py").exists():
    _NB_DIR = next(
        p
        for p in [_NB_DIR / "notebooks" / "audit", *_NB_DIR.parents]
        if (p / "_models.py").exists()
    )
_REPO_ROOT = _NB_DIR.parents[1]
sys.path.insert(0, str(_NB_DIR))
sys.path.insert(0, str(_REPO_ROOT))

import warnings as _w

_w.filterwarnings("ignore", message=".*pynvml.*")  # torch/cuda probe noise
import torch

import torchlens as tl

assert pathlib.Path(tl.__file__).is_relative_to(_REPO_ROOT), (
    f"torchlens imported from {tl.__file__} -- expected this checkout"
)
from _models import ZOO

print(f"torchlens : {tl.__version__}")
print(f"torch     : {torch.__version__}")

## 2. The payoff: sparse vs full capture

A full `tl.trace` saves every tensor; `tl.record(save=pred)` saves only matching ops.
This section makes the contrast concrete.

In [ ]:
# _vocab: tl.record, tl.trace, Recording, Trace

torch.manual_seed(0)
model, x = ZOO["tiny_mlp"]()

# Full trace — everything
full_trace = tl.trace(model, x)

# Sparse recording — only relu ops
rec_relu = tl.record(model, x, save=tl.func("relu"))

# Sparse recording — only linear ops
rec_linear = tl.record(model, x, save=tl.func("linear"))

print("Full trace layers     :", len(full_trace.layer_labels))
print("Recording(relu).n_ops :", rec_relu.n_ops)
print("Recording(linear).n_ops:", rec_linear.n_ops)
print()
print(
    "relu payloads saved  :",
    {k: v.shape for k, v in rec_relu.activation_payloads_by_raw_label.items()},
)
print(
    "linear payloads saved:",
    {k: v.shape for k, v in rec_linear.activation_payloads_by_raw_label.items()},
)

## 3. `Recording` repr and `summary()`

In [ ]:
# _vocab: Recording.__repr__, Recording.summary, Recording.n_ops, Recording.n_records

full_repr = repr(rec_relu)
print(f"repr(rec_relu) is {len(full_repr):,} chars for a ONE-op recording; first 300:")
print(full_repr[:300], "...")
print("⚠️ GAP: a bare `rec` at a REPL floods the screen -- see GAP cell")
print()
print("rec_relu.summary():", rec_relu.summary())
print()
print("n_ops    :", rec_relu.n_ops, "  (ops that matched the save= predicate)")
print("n_records:", rec_relu.n_records, "  (events stored in records list)")

## 4. `Recording.to_pandas()` — tabular event view

In [ ]:
# _vocab: Recording.to_pandas

df_relu = rec_relu.to_pandas()
print("Recording(relu).to_pandas():")
print(df_relu.to_string())
print()
df_linear = rec_linear.to_pandas()
print("Recording(linear).to_pandas():")
print(df_linear.to_string())

## 5. `ActivationRecord` — the low-level record object

Each matched op appears in `rec.records` as an `ActivationRecord` with a `.ctx` context
field and payload attributes. The payload lives in `ram_payload` or `disk_payload`.

In [ ]:
# The records list holds ActivationRecord objects for the matched ops.
# (n_records and len(records) are consistent -- a June-2026 counting mismatch was fixed.)
print("len(rec_relu.records):", len(rec_relu.records))
print("rec_relu.n_records   :", rec_relu.n_records)

# Access activation payloads directly
for raw_label, tensor in rec_relu.activation_payloads_by_raw_label.items():
    print(f"  payload['{raw_label}']: shape={tensor.shape}, dtype={tensor.dtype}")
print()

# Show which predicate was applied + halt state
print("keep_op_repr :", rec_relu.keep_op_repr)
print("halted       :", rec_relu.halted)
print("halt_reason  :", rec_relu.halt_reason)

## 6. `Recording.to_trace()` — cook into a full Trace

`to_trace()` re-runs the captured event stream into a standard `Trace` with all the
normal accessor infrastructure. Unsaved payloads exist as metadata-only ops.

In [ ]:
# _vocab: Recording.to_trace

cooked = rec_relu.to_trace()
print("cooked type     :", type(cooked).__name__)
print("cooked repr     :", repr(cooked))
print("cooked layers   :", cooked.layer_labels)
print()
# All layers exist as metadata; only relu has a payload
print("relu activation via cooked trace:")
print(cooked["relu"].out)

## 7. `RecordingTrace` — the event stream under a Recording

`recording.recording_trace` exposes the full event stream (ops + module enter/exit).
A FRESH recording shows the real content; the notebook's earlier `rec_relu` may read
empty by this point (see the tripwire check below).

In [ ]:
# Tripwire: the notebook-start recording's event stream state at this point
rt_old = rec_relu.recording_trace
first = rt_old.summary().splitlines()[0]
print("rec_relu.recording_trace now:", first)
if "total_events=0" in first:
    print("⚠️ GAP: the event stream of the §2 recording reads EMPTY here, though a fresh")
    print("   recording shows 10 events -- some earlier accessor drained it silently.")
print()

# A fresh recording shows the real event stream
rec_fresh = tl.record(model, x, save=tl.func("relu"))
rt = rec_fresh.recording_trace
print("Fresh recording_trace summary:", rt.summary())
print()
print("RecordingTrace.to_pandas() — full event table including module enter/exit:")
print(rt.to_pandas().to_string())

## 8. `tl.fastlog.dry_run` — pre-flight label discovery

`dry_run` runs the model but saves nothing — it returns only the `RecordingTrace`.
Use it to discover what raw labels exist before committing to a save predicate.

In [ ]:
# _vocab: tl.fastlog.dry_run, RecordingTrace

# dry_run: discover what labels are available (no payloads saved)
dry = tl.fastlog.dry_run(
    model,
    x,
    save=lambda ctx: True,  # accept all so every op appears
)

print("dry_run type   :", type(dry).__name__)
print("dry_run summary:", dry.summary())
print()
print("Full event table (all ops + module events):")
print(dry.to_pandas().to_string())

## 9. `RecordingTrace.repredicate` — offline predicate replay

Already have a `RecordingTrace`? `repredicate(other_keep_op=...)` re-applies a new predicate
without re-running the model — useful for interactive label exploration.

In [ ]:
# _vocab: RecordingTrace.repredicate

rt_linear_only = dry.repredicate(other_keep_op=lambda ctx: ctx.layer_type == "linear")
print("repredicate (linear only) summary:", rt_linear_only.summary())
print()
print("repredicate decisions:")
print(rt_linear_only.to_pandas()[["kind", "op_type", "address", "shape"]].to_string())

## 10. `halt=` — early exit predicate

Returning `True` from `halt=` stops the forward pass immediately after the matching event.
Use it to capture only a prefix of the computation.

In [ ]:
# _vocab: halt=, Recording.halted, Recording.halt_reason

# Halt immediately after the first linear op; save it too
rec_halted = tl.record(
    model,
    x,
    save=tl.func("linear"),
    halt=lambda ctx: ctx.layer_type == "linear",  # stop at first linear
)

print("halted     :", rec_halted.halted)
print("halt_reason:", rec_halted.halt_reason)
print("n_ops (saved before halt):", rec_halted.n_ops)
print()
print("payloads captured before halt:")
for k, v in rec_halted.activation_payloads_by_raw_label.items():
    print(f"  {k}: {v.shape}")

## 11. `tl.fastlog` module — shim attributes and types

In [ ]:
# _vocab: tl.fastlog (module namespace)

public_fastlog = [a for a in dir(tl.fastlog) if not a.startswith("_")]
print("tl.fastlog public attributes:")
for name in sorted(public_fastlog):
    print(f"  {name}")
print()
print("tl.fastlog.record is tl.record:", tl.fastlog.record is tl.record)

## 12. `save=` is the only predicate spelling (aliases removed)

In [ ]:
# The canonical form is save=; the old keep_op= / keep_module= alias kwargs are
# REMOVED (predicate-interpreter consolidation) and now raise TypeError.

rec_canonical = tl.record(model, x, save=tl.func("relu"))
print("canonical save=: keep_op_repr =", rec_canonical.keep_op_repr)
print()

try:
    tl.record(model, x, keep_op=tl.func("relu"))
except TypeError as exc:
    print("removed alias raises:", exc)

## ⚠️ GAPs / ergonomic smells

- *(fixed since 2026-06)* **`Recording.n_records` now agrees with `len(rec.records)`** —
  the stale-zero counting mismatch is gone.
- *(fixed by the predicate consolidation)* `tl.fastlog.dry_run` now accepts `save=`
  and the removed `keep_op=`/`keep_module=` aliases raise TypeError everywhere.
- **`RecordingTrace.repredicate` still uses `other_keep_op=`** — a slot-simulation
  surface that deliberately keeps the internal slot name.
- **`Recording.__repr__` is the full event dump** (~85 KB for a one-op recording — the
  whole RecordContext chain). Interactive `rec` at a REPL floods the screen; a one-line
  summary repr like `summary()` produces would be far kinder.
- **`tl.record` requires a predicate** — `tl.record(model, x)` raises
  `RecordingConfigError`. There is no "record everything" spelling; users must switch
  to `tl.trace`.
- **A `Recording`'s event stream can silently read empty later in a session** — by §7
  the §2 recording's `recording_trace` reports `total_events=0` while a fresh
  recording shows 10; some intermediate accessor drains it without warning (not
  reproducible with `to_trace()` or a second recording alone -- needs a look).
- **Module events appear in `RecordingTrace.to_pandas()` but not `Recording.to_pandas()`**
  — intentional, but should be documented more visibly.